In [ ]:
# ============================================
# KODE LENGKAP DARI AWAL SAMPAI SELESAI
# (Upload ZIP → Extract → Split → Training YOLOv5)
# ============================================

from google.colab import drive
drive.mount('/content/drive')

import os
import zipfile
import shutil
from sklearn.model_selection import train_test_split

Mounted at /content/drive


In [ ]:
# ============================================
# 1. Tentukan LOKASI ZIP DI DRIVE
# ============================================
# GANTI PATH INI dengan lokasi ZIP!
zip_path = '/content/drive/MyDrive/kel.5-compvist/data/mentah.yolov5pytorch(1).zip'

# Folder tujuan ekstrak
extract_path = '/content/drive/MyDrive/kel.5-compvist/hsl.prcbn-EXTRASI/yolo5_diki'

print("📁 Step 1: Ekstrak ZIP...")
# Hapus folder lama kalau ada
if os.path.exists(extract_path):
    shutil.rmtree(extract_path)

# Ekstrak ZIP
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(f"✅ ZIP berhasil diekstrak ke: {extract_path}")

📁 Step 1: Ekstrak ZIP...
✅ ZIP berhasil diekstrak ke: /content/drive/MyDrive/kel.5-compvist/hsl.prcbn-EXTRASI/yolo5_diki


In [ ]:
# ============================================
# 2. CEK STRUKTUR HASIL EKSTRAK
# ============================================
print("\n📁 Step 2: Cek struktur hasil ekstrak...")
!ls -la "{extract_path}"

# Cari tahu struktur sebenarnya (mungkin ada subfolder)
print("\n📁 Cek lebih dalam:")
!find "{extract_path}" -type d -maxdepth 2

# Cari file data.yaml
print("\n📄 Cari file data.yaml:")
!find "{extract_path}" -name "*.yaml"


📁 Step 2: Cek struktur hasil ekstrak...
total 6
-rw------- 1 root root  296 Jun 12 14:15 data.yaml
-rw------- 1 root root  133 Jun 12 14:15 README.dataset.txt
-rw------- 1 root root  834 Jun 12 14:15 README.roboflow.txt
drwx------ 4 root root 4096 Jun 12 14:15 train

📁 Cek lebih dalam:
find: warning: you have specified the global option -maxdepth after the argument -type, but global options are not positional, i.e., -maxdepth affects tests specified before it as well as those specified after it.  Please specify global options before other arguments.
/content/drive/MyDrive/kel.5-compvist/hsl.prcbn-EXTRASI/yolo5_diki
/content/drive/MyDrive/kel.5-compvist/hsl.prcbn-EXTRASI/yolo5_diki/train
/content/drive/MyDrive/kel.5-compvist/hsl.prcbn-EXTRASI/yolo5_diki/train/images
/content/drive/MyDrive/kel.5-compvist/hsl.prcbn-EXTRASI/yolo5_diki/train/labels

📄 Cari file data.yaml:
/content/drive/MyDrive/kel.5-compvist/hsl.prcbn-EXTRASI/yolo5_diki/data.yaml


In [ ]:
# ============================================
# 3. SPLIT DATA (Bikin valid/ kalau belum ada)
# ============================================
print("\n Step 3: Split dataset (80% train, 20% valid)...")

# Coba deteksi struktur yang benar
dataset_root = extract_path

# Cari folder train yang berisi gambar
train_img_path = None
train_lbl_path = None

# Kemungkinan struktur 1: langsung ada train/images
if os.path.exists(f'{dataset_root}/train/images'):
    train_img_path = f'{dataset_root}/train/images'
    train_lbl_path = f'{dataset_root}/train/labels'
    print("Detected: train/images/ dan train/labels/")
# Kemungkinan struktur 2: train/ langsung berisi gambar
elif os.path.exists(f'{dataset_root}/train'):
    files = os.listdir(f'{dataset_root}/train')
    if any(f.endswith(('.jpg', '.png', '.jpeg')) for f in files):
        train_img_path = f'{dataset_root}/train'
        train_lbl_path = f'{dataset_root}/train'
        print("Detected: train/ langsung berisi gambar dan label")
# Kemungkinan struktur 3: ada folder di dalam folder (Unzipped Files)
else:
    for folder in os.listdir(dataset_root):
        subfolder = os.path.join(dataset_root, folder)
        if os.path.isdir(subfolder) and os.path.exists(f'{subfolder}/train/images'):
            train_img_path = f'{subfolder}/train/images'
            train_lbl_path = f'{subfolder}/train/labels'
            dataset_root = subfolder
            print(f"Detected: {folder}/train/images/")
            break

if train_img_path:
    # Ambil semua gambar
    train_images = [f for f in os.listdir(train_img_path) if f.endswith(('.jpg', '.png', '.jpeg'))]
    print(f"Total gambar ditemukan: {len(train_images)}")

    if len(train_images) > 0:
        # Buat folder valid di dataset_root
        valid_img_path = f'{dataset_root}/valid/images'
        valid_lbl_path = f'{dataset_root}/valid/labels'

        os.makedirs(valid_img_path, exist_ok=True)
        os.makedirs(valid_lbl_path, exist_ok=True)

        # Split 80/20
        train_set, valid_set = train_test_split(train_images, test_size=0.2, random_state=42)

        # Pindahkan ke valid
        for img in valid_set:
            shutil.move(
                f'{train_img_path}/{img}',
                f'{valid_img_path}/{img}'
            )
            label = img.rsplit('.', 1)[0] + '.txt'
            if os.path.exists(f'{train_lbl_path}/{label}'):
                shutil.move(
                    f'{train_lbl_path}/{label}',
                    f'{valid_lbl_path}/{label}'
                )

        print(f"Split selesai!")
        print(f"   Train: {len(train_set)} gambar")
        print(f"   Valid: {len(valid_set)} gambar")
    else:
        print("Tidak ada gambar! Cek file ZIP-nya.")
else:
    print("Tidak bisa mendeteksi struktur folder train!")


 Step 3: Split dataset (80% train, 20% valid)...
Detected: train/images/ dan train/labels/
Total gambar ditemukan: 174
Split selesai!
   Train: 139 gambar
   Valid: 35 gambar


In [ ]:
# ============================================
# 4. INSTALL YOLOv5
# ============================================
print("\n📁 Step 4: Install YOLOv5...")
if not os.path.exists('/content/yolov5'):
    !git clone https://github.com/ultralytics/yolov5
%cd /content/yolov5
!pip install -qr requirements.txt


📁 Step 4: Install YOLOv5...
Cloning into 'yolov5'...
remote: Enumerating objects: 18320, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 18320 (delta 26), reused 20 (delta 11), pack-reused 18264 (from 2)
Receiving objects: 100% (18320/18320), 17.45 MiB | 18.24 MiB/s, done.
Resolving deltas: 100% (12444/12444), done.
/content/yolov5
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 4.1 MB/s eta 0:00:00


In [ ]:
# ============================================
# 5 COPY DATASET KE DALAM FOLDER YOLOv5
# ============================================

# Path dataset asli di Drive
source = '/content/drive/MyDrive/kel.5-compvist/hsl.prcbn-EXTRASI/yolo5_diki'

# Tujuan di folder YOLOv5
destination = '/content/yolov5/datasets/banana'

# Hapus folder tujuan kalau sudah ada
if os.path.exists(destination):
    shutil.rmtree(destination)

# Copy dataset
shutil.copytree(source, destination)

print(f"✅ Dataset sudah di-copy ke: {destination}")
print("📁 Struktur:")
!ls -la "{destination}"

✅ Dataset sudah di-copy ke: /content/yolov5/datasets/banana
📁 Struktur:
total 28
drwx------ 4 root root 4096 Jun 12 14:17 .
drwxr-xr-x 3 root root 4096 Jun 12 14:20 ..
-rw------- 1 root root  296 Jun 12 14:15 data.yaml
-rw------- 1 root root  133 Jun 12 14:15 README.dataset.txt
-rw------- 1 root root  834 Jun 12 14:15 README.roboflow.txt
drwx------ 4 root root 4096 Jun 12 14:15 train
drwx------ 4 root root 4096 Jun 12 14:17 valid


In [ ]:
# ============================================
# 6 PERBAIKI FILE data.yaml (SETELAH DI-COPY)
# ============================================
import yaml

data_yaml_path = '/content/yolov5/datasets/banana/data.yaml'

# Baca file data.yaml
with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

# Ubah path train dan valid menjadi ABSOLUT (bukan relatif)
data_config['train'] = '/content/yolov5/datasets/banana/train/images'
data_config['val'] = '/content/yolov5/datasets/banana/valid/images'

# Simpan kembali
with open(data_yaml_path, 'w') as f:
    yaml.dump(data_config, f)

print("✅ data.yaml sudah diperbaiki!")
print(f"train: {data_config['train']}")
print(f"val: {data_config['val']}")

# Cek isinya
print("\n📄 Isi data.yaml sekarang:")
!cat "{data_yaml_path}"

✅ data.yaml sudah diperbaiki!
train: /content/yolov5/datasets/banana/train/images
val: /content/yolov5/datasets/banana/valid/images

📄 Isi data.yaml sekarang:
names:
- busuk
- matang
- mentah
nc: 3
roboflow:
  license: CC BY 4.0
  project: mentah-askyy
  url: https://universe.roboflow.com/dicky-triharyadi/mentah-askyy/dataset/dataset
  version: dataset
  workspace: dicky-triharyadi
test: ../test/images
train: /content/yolov5/datasets/banana/train/images
val: /content/yolov5/datasets/banana/valid/images


In [ ]:
# ============================================
# TRAINING YOLOv5
# ============================================
%cd /content/yolov5

!python train.py \
    --img 640 \
    --batch 16 \
    --epochs 20 \
    --data /content/yolov5/datasets/banana/data.yaml \
    --weights yolov5s.pt \
    --device cpu

print("✅ Training YOLOv5 selesai!")

/content/yolov5
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
wandb: WARNING ⚠️ wandb is deprecated and will be removed in a future release. See supported integrations at https://github.com/ultralytics/yolov5#integrations.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice: (30 second timeout) 
wandb: WARNING W&B disabled due to login timeout.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
train: weights=yolov5s.pt, cfg=, data=/content/yolov5/datasets/banana/data.yaml, hyp=data/hyps/hyp.scratch-low.yaml, epochs=20, batch_size=16, imgsz=640, rect=False, resume=False, nosave=False, noval=False, noautoanchor=False, 

In [ ]:
%cd /content/yolov5

!python train.py \
    --img 640 \
    --batch 16 \
    --epochs 20 \
    --data /content/yolov5/datasets/banana/data.yaml \
    --weights /content/yolov5/runs/train/exp/weights/last.pt \
    --device cpu \
    --resume

/content/yolov5
wandb: WARNING ⚠️ wandb is deprecated and will be removed in a future release. See supported integrations at https://github.com/ultralytics/yolov5#integrations.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice: (30 second timeout) 
wandb: WARNING W&B disabled due to login timeout.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
train: weights=/content/yolov5/runs/train/exp/weights/last.pt, cfg=, data=/content/yolov5/datasets/banana/data.yaml, hyp=data/hyps/hyp.scratch-low.yaml, epochs=20, batch_size=16, imgsz=640, rect=False, resume=True, nosave=False, noval=False, noautoanchor=False, noplots=False, evolve=None, evolve_population=data/hyps, resume_evolve=None, bucket=, cache=None, image_weights=False, device=cpu, multi_scale=False, single_cls=False, optimizer=SGD, sync_bn=False, workers=8, project=runs/train, name=exp, exist_ok=False, quad=False, cos_lr=Fa

In [ ]:
# ============================================
# COPY HASIL TRAINING exp3 KE DRIVE
# ============================================

source = '/content/yolov5/runs/train/exp'
destination = '/content/drive/MyDrive/kel.5-compvist/hasil_prcbn/kel5_yolo5'

# Hapus folder tujuan kalau sudah ada
if os.path.exists(destination):
    shutil.rmtree(destination)
    print("🗑️ Folder lama dihapus")

# Copy folder exp3
shutil.copytree(source, destination)

print("\n✅ HASIL TRAINING BERHASIL DISIMPAN KE DRIVE!")
print(f"📍 Lokasi: {destination}")

# ============================================
# VERIFIKASI
# ============================================
print("\n📁 Cek folder weights di Drive:")
!ls -la "{destination}/weights/"

print("\n✅ best.pt dan last.pt sudah aman di Drive!")

🗑️ Folder lama dihapus

✅ HASIL TRAINING BERHASIL DISIMPAN KE DRIVE!
📍 Lokasi: /content/drive/MyDrive/kel.5-compvist/hasil_prcbn/kel5_yolo5

📁 Cek folder weights di Drive:
total 28043
-rw------- 1 root root 14357935 Jun 12 15:53 best.pt
-rw------- 1 root root 14357935 Jun 12 15:53 last.pt

✅ best.pt dan last.pt sudah aman di Drive!


In [ ]:
# Path best.pt
best_path = '/content/drive/MyDrive/kel.5-compvist/hasil_prcbn/kel5_yolo5/weights/best.pt'

if os.path.exists(best_path):
    size = os.path.getsize(best_path) / (1024*1024)
    print(f"✅ best.pt ADA di Drive!")
    print(f"📍 Lokasi: {best_path}")
    print(f"📦 Ukuran: {size:.2f} MB")

    # Cek file lain yang ada di folder yang sama
    print("\n📁 Isi folder weights:")
    !ls -la "{os.path.dirname(best_path)}/"
else:
    print(f"❌ best.pt TIDAK DITEMUKAN!")

✅ best.pt ADA di Drive!
📍 Lokasi: /content/drive/MyDrive/kel.5-compvist/hasil_prcbn/kel5_yolo5/weights/best.pt
📦 Ukuran: 13.69 MB

📁 Isi folder weights:
total 28043
-rw------- 1 root root 14357935 Jun 12 15:53 best.pt
-rw------- 1 root root 14357935 Jun 12 15:53 last.pt


In [ ]:
# Test load model untuk memastikan file tidak corrupt
from ultralytics import YOLO

best_path = '/content/drive/MyDrive/kel.5-compvist/hasil_prcbn/kel5_yolo5/weights/best.pt'

try:
    model = YOLO(best_path)
    print("✅ Model BERHASIL di-load!")
    print(f"🏷️ Kelas yang dideteksi: {model.names}")
    print(f"📊 Jumlah kelas: {len(model.names)}")
except Exception as e:
    print(f"❌ Gagal load model: {e}")

✅ Model BERHASIL di-load!
🏷️ Kelas yang dideteksi: {0: 'busuk', 1: 'matang', 2: 'mentah'}
📊 Jumlah kelas: 3
